In [ ]:
from torchaudio.models.decoder import ctc_decoder
import numpy as np 
import torch
from torch.nn.functional import softmax, log_softmax
from brainaudio.inference.eval_metrics import _cer_and_wer
from brainaudio.inference.lm_funcs import normalize_shorthand
import pandas as pd

In [ ]:
language_model_path = "/data2/brain2text/lm/"
units_txt_file_pytorch = f"{language_model_path}units_pytorch.txt"
imagineville_vocab_phoneme = f"{language_model_path}vocab_lower_100k_pytorch_phoneme.txt"

In [ ]:
with open(units_txt_file_pytorch, "r", encoding="utf-8") as f:
    # Use .strip() to remove the newline character from every line
    units = [line.strip() for line in f]
    
units[24]

In [ ]:
decoder = ctc_decoder(tokens=units_txt_file_pytorch, lexicon=imagineville_vocab_phoneme, 
                       beam_size=300, nbest=20, lm="/data2/brain2text/lm/lm_dec19_huge_4gram.kenlm", 
                       lm_weight=2.0, word_score=0.1, log_add=False, sil_score=0, beam_threshold=1e3)

In [ ]:
val_transcripts_24 = pd.read_pickle("/data2/brain2text/b2t_24/transcripts_val_cleaned.pkl")
val_transcripts_25 = pd.read_pickle("/data2/brain2text/b2t_25/transcripts_val_cleaned.pkl")

In [ ]:
# model_logits = np.load("/data2/brain2text/b2t_25/logits/pretrained_RNN/logits_val.npz")
base_path_b2t_25 = "/data2/brain2text/b2t_25/logits/"

LOGITS_VAL_PATH_ARR = [f"{base_path_b2t_25}best_chunked_transformer_combined_seed_0/logits_val_chunk:5_context:20.npz",
                       f"{base_path_b2t_25}pretrained_RNN/logits_val.npz"]

model_logits = np.load(LOGITS_VAL_PATH_ARR[1])

In [ ]:
acoustic_scale = 0.6
ground_truth_arr = []
pred_arr = []
for idx in range(len(model_logits)):
    if idx % 100 == 0:
        print(idx)
    single_trial_logits = torch.as_tensor(model_logits[f'arr_{idx}']).float().unsqueeze(0)
    beam_search_char = decoder(single_trial_logits * acoustic_scale)
    beam_search_transcript_char = normalize_shorthand(" ".join(beam_search_char[0][0].words).strip())
    ground_truth_sentence = val_transcripts_25[idx]
    ground_truth_arr.append(ground_truth_sentence)
    pred_arr.append(beam_search_transcript_char)
    
cer, wer, wer_sent = _cer_and_wer(pred_arr, ground_truth_arr)
print(wer)

In [ ]:
# baseline RNN WER = 0.1176, Decoding time = 3 mimnutes and 6.4 seconds (on val set of B2T_25)
# transformer combined chunked WER = 0.09179, Decoding time = 3 mimnutes and 0.3 seconds (on val set of B2T_25)

# So we observe about a two percent decrease in WER when using the chunked transformer model over the baseline RNN model.
# When using the custom beam search with an LLM, I am curently getting at most a 1 perecent decrease in WER when switching 
# from the baseline GRU to the chunked transformer.

# Why is this occurring?

In [ ]:
for p, g in zip(pred_arr, ground_truth_arr):
    
    print(g, " ||||  ", p)

In [ ]:
lm_weights = [1.5, 2, 2.5, 3]   
acoustic_score = [0.5, 0.6, 0.7, 0.8, 0.9]
word_penalty = [0, 0.1, -0.1, 0.2, -0.2, 0.3, -0.3]
sil_score = [0, -0.1, -0.2, -0.3]
beam_size = [100]

wer_dict = {}
wer_dict['sil'] = []
wer_dict['lmw'] = []
wer_dict['wp'] = []
wer_dict['bs'] = []
wer_dict['ac'] = []

wer_dict['wer'] = []

for sil in sil_score:
    for wp in word_penalty:
      for lmw in lm_weights:
          for bs in beam_size:
          
            decoder = ctc_decoder(tokens=units_txt_file_pytorch, lexicon=imagineville_vocab_phoneme, 
                       beam_size=bs, nbest=1, lm="/data2/brain2text/lm/lm_dec19_huge_4gram.kenlm", 
                       lm_weight=lmw, word_score=wp, sil_score=sil, log_add=True, beam_threshold=1e3)

            for ac in acoustic_score:
              
              ground_truth_arr = []
              pred_arr = []
          
              for idx in range(880):
                  single_trial_logits = torch.as_tensor(model_logits[f'arr_{idx}']).float().unsqueeze(0)
                  beam_search_outs = decoder(single_trial_logits*ac)
                  beam_search_transcript = normalize_shorthand(" ".join(beam_search_outs[0][0].words).strip())
                  ground_truth_sentence = val_transcripts[idx]
                  ground_truth_arr.append(ground_truth_sentence)
                  pred_arr.append(beam_search_transcript)
                  
              cer, wer, wer_sent = _cer_and_wer(pred_arr, ground_truth_arr)
              
              
              wer_dict['sil'].append(sil)
              wer_dict['lmw'].append(lmw)
              wer_dict['wp'].append(wp)
              wer_dict['bs'].append(bs)
              wer_dict['ac'].append(ac)
              wer_dict['wer'].append(wer)
              
              
              print(f"sil score: {sil}, lm weight: {lmw}, word penalty: {wp}, beam size: {bs}, acoustic score: {ac},  wer: {wer}")
